# Exploring Accidents

In [ ]:
import pandas as pd
import sqlite3

In [2]:
conn = sqlite3.connect("vic_crash_data.db")

In [13]:
accidents = pd.read_sql("select * from accidents", conn)
accidents.head()

,accident_no,accident_date,accident_time,accident_type,day_of_week,dca_code,node_id,no_of_vehicles,no_killed,no_person_inj2,no_person_inj3,no_person_not_inj,police_attended,road_geometry,light_condition,severity,speed_zone,rma
0,T20120004516,2012-02-24,15:30:00.000000,2,6,101,249991,1,0,0,1,1,1,5,1,3,50,Local Road
1,T20120007847,2012-04-03,01:57:00.000000,1,3,130,251467,2,0,0,1,2,1,5,5,3,50,Local Road
2,T20120008205,2012-04-07,01:15:00.000000,2,7,105,251982,1,0,1,0,1,1,5,3,2,80,Arterial Highway
3,T20120008420,2012-02-16,09:15:00.000000,8,5,174,251094,1,0,0,1,0,2,5,1,3,60,Freeway
4,T20120008442,2012-04-10,10:00:00.000000,1,3,121,45652,2,0,1,2,0,1,2,1,2,70,Arterial Other


## Starting and End time data

In [9]:
print(f" Data is from {accidents["accident_date"].min()} to {accidents["accident_date"].max()}")

 Data is from 2012-01-01 to 2025-05-31


## Total number of Accidents

In [11]:
print(f"Total accidents: {len(accidents):,}")

Total accidents: 191,779


In [19]:
accidents.isna().sum()

accident_no             0
accident_date           0
accident_time           0
accident_type           0
day_of_week             0
dca_code                0
node_id                 0
no_of_vehicles          0
no_killed               0
no_person_inj2          0
no_person_inj3          0
no_person_not_inj       0
police_attended         0
road_geometry           0
light_condition         0
severity                0
speed_zone              0
rma                  8751
year                    0
dtype: int64

## Accidents per year

In [20]:
accidents["year"] = pd.to_datetime(accidents["accident_date"]).dt.year
accidents.groupby("year")["accident_no"].count()

year
2012    13912
2013    14023
2014    14503
2015    15684
2016    15779
2017    13351
2018    12765
2019    14397
2020    12169
2021    14013
2022    15034
2023    15532
2024    14632
2025     5985
Name: accident_no, dtype: int64

## Type of Accidents

In [21]:
type_of_accidents = pd.read_sql("""
select a.accident_no, d.accident_type_desc from accidents a
left join accident_type_desc d on a.accident_type = d.accident_type
""", conn)
type_of_accidents.head()

,accident_no,accident_type_desc
0,T20120004516,Struck Pedestrian
1,T20120007847,Collision with vehicle
2,T20120008205,Struck Pedestrian
3,T20120008420,No collision and no object struck
4,T20120008442,Collision with vehicle


In [25]:
type_of_accidents.groupby("accident_type_desc")["accident_no"].count().sort_values()

accident_type_desc
Other accident                          209
Fall from or in moving vehicle         1240
collision with some other object       2012
Struck animal                          2095
No collision and no object struck      7925
Vehicle overturned (no collision)      9017
Struck Pedestrian                     15790
Collision with a fixed object         28797
Collision with vehicle               124694
Name: accident_no, dtype: int64

## DCA Code (Definition of Classifying Accidents)

In [31]:
accidents["dca_code"].nunique()

81

## Road geometry and Accidents

In [32]:
road_geometry = pd.read_sql("""
select a.accident_no, r.road_geometry_desc from accidents a
left join road_geometry r on a.accident_type = r.road_geometry
""", conn)
road_geometry.head()

,accident_no,road_geometry_desc
0,T20120004516,T intersection
1,T20120007847,Cross intersection
2,T20120008205,T intersection
3,T20120008420,Private property
4,T20120008442,Cross intersection


In [34]:
road_geometry.groupby("road_geometry_desc")["accident_no"].count().sort_values()

road_geometry_desc
Unknown                     209
Road closure               1240
Not at intersection        2012
Y intersection             2095
Private property           7925
Dead end                   9017
T intersection            15790
Multiple intersection     28797
Cross intersection       124694
Name: accident_no, dtype: int64

## Target Variable - Severity

In [37]:
accidents.groupby("severity")["accident_no"].count()

severity
1      3207
2     67329
3    121239
4         4
Name: accident_no, dtype: int64

In [ ]:
accidents[accidents["severity"] == '1'].groupby("year")["accident_no"].count()

year
2012    261
2013    225
2014    223
2015    231
2016    275
2017    240
2018    202
2019    248
2020    195
2021    216
2022    239
2023    261
2024    271
2025    120
Name: accident_no, dtype: int64

## Severity and Accident Time

In [41]:
accidents["hour"] = pd.to_datetime(accidents["accident_time"], format = '%H:%M:%S.%f').dt.hour
accidents.head()

,accident_no,accident_date,accident_time,accident_type,day_of_week,dca_code,node_id,no_of_vehicles,no_killed,no_person_inj2,no_person_inj3,no_person_not_inj,police_attended,road_geometry,light_condition,severity,speed_zone,rma,year,hour
0,T20120004516,2012-02-24,15:30:00.000000,2,6,101,249991,1,0,0,1,1,1,5,1,3,50,Local Road,2012,15
1,T20120007847,2012-04-03,01:57:00.000000,1,3,130,251467,2,0,0,1,2,1,5,5,3,50,Local Road,2012,1
2,T20120008205,2012-04-07,01:15:00.000000,2,7,105,251982,1,0,1,0,1,1,5,3,2,80,Arterial Highway,2012,1
3,T20120008420,2012-02-16,09:15:00.000000,8,5,174,251094,1,0,0,1,0,2,5,1,3,60,Freeway,2012,9
4,T20120008442,2012-04-10,10:00:00.000000,1,3,121,45652,2,0,1,2,0,1,2,1,2,70,Arterial Other,2012,10


In [45]:
time_severity = accidents[accidents["severity"] == '1'].groupby("hour")["accident_no"].count()
time_severity.head()

hour
0    91
1    77
2    72
3    65
4    49
Name: accident_no, dtype: int64